# Task

Design an in-memory file system that supports creating files and directories, navigating paths, and basic file operations like move and rename

### Clarifying Questions

Q: What kind of characters are allowed in the file and directory names?

For this problem, you can assume names consist of alphanumeric characters, hyphens, underscores, and dots — no spaces or special characters like slashes. Names are also case-sensitive.

Q: For file operations, besides move and rename, can user delete files?

Yes, delete is in scope — users can delete both files and directories. Note that deleting a non-empty directory should also remove all of its contents recursively.

Q: same operations apply to directories?

Yes, the same operations apply to directories — you can create, delete, list, rename, and move directories, subject to the same validity rules (e.g., you cannot move a directory into one of its own descendants).

Q: A directory can contain both subdirectories and a bunch of files?

Yes, that's correct — a directory can contain both subdirectories and files at the same level.

Q: Can you clarify what is needed for navigating path?

"Navigating a path" means resolving a given Unix-style absolute path string (e.g., /home/user/docs) to the actual file or directory it refers to in the in-memory system. All paths start from the root (/), and your system should be able to locate the correct entry by traversing each component of the path in sequence. You don't need to support relative paths or path shortcuts like . or ...

Q: So given a string (path), the method should return a reference to the actual file or directory?

Yes, that's correct — given an absolute path string, the system should be able to resolve and return a reference to the corresponding file or directory entry in memory.

Q: what happens when user tries to name a file or directory with a name that already exists in the same directory?

That's a name collision — and yes, it should be treated as an invalid operation. If a file or directory already exists with the same name in the same parent directory, the operation should fail and throw a specific exception.

Q: Is there a limit on the length of names?

For this problem, you can assume there's no enforced limit on name length — don't worry about that constraint.

Q: Do I need to support bulk moving of files and directories?

No, bulk operations are out of scope — each move operation applies to a single file or directory at a time.

Q: how about bulk delete?

Bulk delete is out of scope — each delete operation applies to a single file or directory at a time.

Q: Do I need to create actual file or directory on disk?

No — this is a purely in-memory file system. Nothing should be written to or read from disk; all data exists only in memory for the lifetime of the program.

Q: How about UI and logging?

Both are out of scope for this problem — no UI is needed, and you don't need to implement any logging infrastructure.

Q: is there any limit on the number of files and subdirectories a directory can contain?

No enforced limit — a directory can contain any number of files and subdirectories. You can assume memory is sufficient to handle tens of thousands of entries across the entire file system.

Q: if user tries to navigate to a non exist directory or file, what should be the respond?

If a user tries to navigate to a path that doesn't exist, the system should throw a specific exception indicating that the path was not found.

Q: since there is no UI, when user wants to perform an operation, is she going to input the path of the file or directory for the operation?

Yes — since there's no UI, you can think of the public API of your file system as a set of methods that accept path strings as parameters. For example, an operation like delete or move would take the absolute path(s) of the target entry as input.

Q: Names are case sensitive, so two files with the same name but different cases are allowed?

Yes, that's correct — since names are case-sensitive, file.txt and File.txt would be treated as two distinct entries and could coexist in the same directory without conflict.

Q: If name collision happens during a move operation, what should be expected?

If a name collision occurs during a move operation — meaning an entry with the same name already exists in the destination directory — the operation should fail and throw a specific exception, just like it would for any other name collision.

Q: Does calling list() on a folder return a list of names for the files and subfolders within it or the actual objects?

For this problem, list() can return just the names — a list of strings representing the names of the direct children (files and subdirectories) within that directory.

Q: In that case, how to distinguish a file from a folder? Can both have the same name?

Good question — within the same directory, a file and a folder cannot share the same name, since names must be unique among all direct children regardless of type. As for distinguishing them in the list() output, that's a design decision for you to make.

# 1. Requirements

### My Answer - 1st attempt
1. An in-memory file system that contains files and directories
2. User can create files or directories at the root level or under any existing directories.
3. A directory can contain both files and subdirectories at the same level. No limit on the
    number of files or subdirectories.
4. File and directory names can contain alphanumeric characters, hyphens, underscores, and dots. 
    All other characters are invalid. Names are case sensitive. No limit on name length.
5. User can perform these actions on a file or directory: navigate, rename, move or delete.
    All operations accept path strings as input.
6. User can also list the content of a directory.
7. Deleting a non-empty directory also removed all of its contents recursively.
8. Raise a specific exception for any of these error conditions:
    - invalid names when creating or renaming a file or directory
    - name collision when creating, renaming or moving a file or directory
    - perform actions on non-exist files or directories
    - moving a directory to its own descendants
    - moving a file or directory to a destination that is a file (not directory) 
    - list operation on a file


Out of Scope:
- Bulk operations
- Support relative paths or path shortcuts
- Files or directories written or read from disk
- UI or logging

### Feedback

What went well

You covered all the primary operations clearly — create, delete, list, rename, move, and path resolution — and correctly noted that all operations accept path strings as input, which shows you internalized the API design well. 
Your error handling section is genuinely strong. You caught name collisions across multiple operations, moving a directory into its own descendant, navigating to non-existent paths, and even the edge case of trying to list a file — that last one especially shows sharp thinking. 
Your out-of-scope section is well-considered and keeps the design tightly focused on what was actually discussed. 

What needs improvement

One error case worth adding is what happens when a user tries to delete the root directory — that's a meaningful invalid operation that your system should handle with a specific exception, and it's easy to overlook until you're mid-implementation. 
You mentioned that creating files or directories works 'at the root level or under any existing directories,' but it would be worth explicitly calling out that creating an entry at a path where the parent directory doesn't exist should also throw a specific exception — right now your error handling section doesn't cover that case. 

Minor suggestions

You might consider briefly noting that files store string content — it's a small detail, but it helps clarify the data model when you move into class design and someone asks what a file actually holds. 

### My Answer - 2nd attempt
1. An in-memory file system that contains files and directories
2. User can create files or directories at the root level or under any existing directories.
3. A directory can contain both files and subdirectories at the same level. No limit on the
    number of files or subdirectories.
4. A file can store string content.    
5. File and directory names can contain alphanumeric characters, hyphens, underscores, and dots. 
    All other characters are invalid. Names are case sensitive. No limit on name length.
6. User can perform these actions on a file or directory: navigate, rename, move or delete.
    All operations accept path strings as input.
7. User can also list the content of a directory.
8. Deleting a non-empty directory also removed all of its contents recursively.
9. Raise a specific exception for any of these error conditions:
    - invalid names when creating or renaming a file or directory
    - name collision when creating, renaming or moving a file or directory
    - creating a file or directory under a parent directory that does not exist  
    - perform rename, move, or delete on non-exist files or directories
    - moving a directory to its own descendants
    - moving a file or directory to a destination that is a file (not directory) 
    - moving a file or directory to a non-exist destination
    - list operation on a file
    - list operation on a non-exist directory
    - deleting, moving, or renaming the root directory
    

Out of Scope:
- Bulk operations
- Support relative paths or path shortcuts
- Files or directories written or read from disk
- UI or logging

### Feedback

What went well

You addressed both pieces of feedback from last time: you added the root directory protection ('deleting, moving, or renaming the root directory') and explicitly called out that creating an entry where the parent doesn't exist throws an exception — those are exactly the gaps we discussed, and it's great to see them covered cleanly. 

Your error handling section is remarkably thorough. You caught name collisions across create, rename, and move; moving a directory into its own descendants; listing a file instead of a directory; and navigating to non-existent paths — these edge cases often trip people up mid-implementation, so having them written down now is a real asset. 

Your out-of-scope section is tight and well-reasoned — bulk operations, relative paths, persistence, and UI are all correctly excluded, keeping the design focused on what actually matters here. 


### Sample Answer

1. Hierarchical file system with single root directory
2. Files store string content
3. Folders contain files and other folders
4. Create and delete files and folders
5. List contents of a folder
6. Navigate/resolve absolute paths (e.g., /home/user/docs)
7. Rename and move files and folders
8. Retrieve full path from any file/folder reference
9. Scale to tens of thousands of entries in memory

Out of Scope:
- Search functionality
- Relative path resolution (../ or ./)
- Permissions, ownership, timestamps
- File type-specific behavior
- Persistence / disk storage
- Symbolic links
- UI layer

# 2. Entities and Relationships

### My Answer - Entities
- FileSystem (FileManager)
- File
- Directory

### Feedback

What went well

You nailed all three core entities — FileSystem, File, and Directory cover exactly what this system needs to model. 
Nice touch including FileManager as an alternate name for FileSystem — shows you're already thinking about how this thing gets used as an API entry point. 

### My Answer - Relationships (1st attempt)

FileManager is the entry point and orchestrator of the system. It contains a reference to the root directory. 
It also owns validation and enforces constraints and rules. All operations (create, rename, delete, list, navigate) have to be through this class.

File is a data holder for properties like name, and content.

Directory is a data holder for name and list of files and subdirectories within it.

### Feedback (1st attempt)

What went well

You clearly identified FileManager as the orchestrator — explaining that all operations flow through it and that it enforces constraints is exactly the right framing. 
You showed the key ownership relationships well: FileManager owns the root directory, Directory holds a list of files and subdirectories, and File holds its content. 

Minor suggestions

One thing worth thinking about: since both File and Directory share common properties like 'name', it could be useful to mention whether they share a common base type or interface — that would help clarify how Directory's children list can hold both files and subdirectories together. 

### My Answer - Relationships (2nd attempt)

FileManager is the entry point and orchestrator of the system. It contains a reference to the root directory. 
It also owns validation and enforces constraints and rules. All operations (create, rename, delete, list, navigate) have to be through this class.

File is a data holder for properties like name, and content.

Directory is a data holder for name and list of files and subdirectories within it.

The File and Directory class can share the same base class or interface, so that the children list in the Directory class can hold objects from both classes. 

### Feedback (2nd attempt)

What went well

You clearly identified FileManager as the orchestrator — explaining that all operations flow through it and that it enforces validation and constraints is exactly the right framing for a facade/entry-point pattern. 

You addressed the previously flagged gap directly: your explanation that File and Directory can share a base class or interface so that the Directory's children list can hold both types shows solid understanding of how the composite structure fits together. 

The ownership chain is well laid out — FileManager holds the root Directory, Directory holds a list of files and subdirectories, and File holds its own content and name. 

### Sample Answer

"FileSystem is the Facade - it owns the root Folder and provides the public API for all path-based operations. File and Folder both inherit from FileSystemEntry, sharing name, parent pointer, and getPath() behavior (Composite pattern). Each Folder has a Map of children keyed by name, containing any mix of Files and other Folders. Each entry stores a parent pointer to its containing Folder, enabling dynamic path computation by walking up the tree. Files are leaf nodes with content, Folders are container nodes with children."

# 3. Class Design

The FileSystem class has an internal reference to the root folder. It offers public APIs for all file and folder operations.

The design uses composite pattern. Internal objects like File and Folder implement the abstract class FileSystemItem.
 
The APIs return FileSystemMetadata which are implemented as immutable objects for Files or Folders. This way, users cannot modify the internal objects directly. This is to adhere to the requirement that the FileSystem class is the single entry point for all operations.  

 

```
ENUM ItemType:
    FILE
    FOLDER

class FileSystem:
    - _root: Folder
    + FileSystem()
    + creatFile(path, content) -> FileMetadata
    + createFolder(path) -> FolderMetadata
    + delete(path)
    + _getParentAndChild(path) -> Folder, FileSystemItem
    + get(path) -> FileSystemMetadata
    + _isNameValid(name) -> bool
    + list(path) -> List<FileSystemMetadata>
    + move(sourcePath, destinationPath)
    + rename(newName, path)

abstract class FileSystemItem:
    - _name: string
    + getName() -> string
    + setName(name)
    + getMetadata() -> FileSystemMetadata
    + isDirectory() -> bool

class File implement FileSystemItem:
    - _content: string?
    + File(name, content)
    + getContent() -> string
    + setContent(content)
    
class Folder implement FileSystemItem:
    - _children: Map<string, FileSystemItem>
    + Folder(name)
    + addChild(FileSystemItem)
    + getChild(name) -> FileSystemItem
    + getChildren() -> List<FileSystemItem>
    + hasChild(name) -> bool
    + removeChild(name)
    
abstract class FileSystemMetadata:
    - name: string
    - type: ItemType
    + FileSystemMetadata(name, itemType)
    + getName() -> string
    + getItemType() -> ItemType

class FileMetadata implement FileSystemMetadata:
    - _content: string?
    + FileMetadata(name, content)
    + getContent() -> string
    
class FolderMetadata implement FileSystemMetadata:
    - _childCount: int
    + FolderMetadata(name, childCount)
    + getChildCount() -> int

```    


### Feedback
What went well

You nailed the Composite pattern with FileSystemItem as the shared abstraction for File and Folder, and your verbal explanation showed you understand exactly why that structure works here. 
Your Folder using a Map<string, FileSystemItem> for _children is exactly the right call — O(1) lookups will matter at scale, and you clearly thought about that. 
The FileSystem class is cleanly designed as a facade with all the right public operations: createFile, createFolder, delete, list, move, rename, and get. That's complete coverage of the required behaviors. 
What needs improvement

Your FileSystemItem (and by extension File and Folder) is missing a parent pointer back to the containing Folder. Without it, computing a full path like /home/user/docs requires the FileSystem to track parent relationships externally — but with a parent pointer, any entry can reconstruct its own path by walking up the tree, which makes move and rename much cleaner to implement. 
Minor suggestions

Since Folder.addChild() and removeChild() will need to maintain the bidirectional link (setting and clearing the parent pointer on the child), it's worth thinking now about how those two methods will coordinate that — it's easy to forget the cleanup side in removeChild. 


### Sample Answer

"This design uses the Composite pattern (FileSystemEntry as shared base, File as leaf, Folder as composite) and the Facade pattern (FileSystem as the single entry point). Path resolution and tree navigation are encapsulated in FileSystem. Parent pointers enable O(depth) path computation without cascading updates on rename/move. Folder uses a Map for O(1) child lookups."

```
abstract class FileSystemEntry:
    - name: string
    - parent: Folder?

    + FileSystemEntry(name)
    + getName() -> string
    + setName(name)
    + getParent() -> Folder?
    + setParent(Folder?)
    + getPath() -> string
    + isDirectory() -> boolean  // abstract


class File extends FileSystemEntry:
    - content: string

    + File(name, content)
    + getContent() -> string
    + setContent(content)
    + isDirectory() -> false


class Folder extends FileSystemEntry:
    - children: Map<string, FileSystemEntry>

    + Folder(name)
    + isDirectory() -> true
    + addChild(entry) -> boolean
    + removeChild(name) -> FileSystemEntry?
    + getChild(name) -> FileSystemEntry?
    + hasChild(name) -> boolean
    + getChildren() -> List<FileSystemEntry>


class FileSystem:
    - root: Folder

    + FileSystem()
    + createFile(path, content) -> File
    + createFolder(path) -> Folder
    + delete(path)
    + list(path) -> List<FileSystemEntry>
    + get(path) -> FileSystemEntry
    + rename(path, newName)
    + move(srcPath, destPath)

```

# 4. Implementation
Implement _getParentAndChild(path) — given an absolute path string like /home/user/docs/file.txt, parse it into segments, walk the Folder tree from _root, and return both the parent Folder and the target FileSystemItem. What exceptions do you raise if an intermediate segment doesn't exist, or if an intermediate segment is a File instead of a Folder?

### My Answer
I have provided the core function here. I can add more error checking later if needed, like checking for invalid characters in path.

The function first check if the input path is 'root'. If it is, return the root folder without any parent. It then iterates over each item in the path. For each one (except the last item), the method will check if the item is a directory and if it contains the item at the next level. If either one fails, a specific error will be raised. 

### Feedback
What went well

Your happy-path walk is solid — you correctly special-case the root, split segments, and return both the parent and target child at the last index. 
You correctly guard against an intermediate segment being a File by checking parent.isDirectory() before descending, raising a specific error. 
What needs improvement

In your empty-items check you wrote return InvalidPathError instead of raise InvalidPathError, so it would return the class rather than signal the error (though this branch is hard to reach in practice). 

Minor suggestions

Consider whether a missing intermediate directory and a missing final child should raise the same ItemNotExistError, or whether an intermediate-that-is-a-File deserves a distinct message from a truly missing path for clearer debugging. 

In [ ]:
    def _getParentAndChild(self, path: str) -> tuple[Folder, FileSystemItem]:
        if path == self.ROOT_PATH: # root has no parent
            return None, self._root
                
        items = path.strip("/").split("/")
        if not items:
            return InvalidPathError
        
        parent = self._root
        for i, name in enumerate(items):
            name = name.strip()
            if not name:
                raise InvalidPathError
            if not parent.isDirectory():
                raise InvalidPathError
            if not parent.hasChild(name):
                raise ItemNotExistError(name)
            
            if i == len(items)-1:
                return parent, parent.getChild(name)
            parent = parent.getChild(name)

        raise ItemNotExistError(path)

Implement move(sourcePath, destinationPath) — this is the most complex operation. 
Walk through how you'd: 
(1) resolve both paths, 
(2) detect if the destination is a descendant of the source (to prevent moving a folder into itself), 
(3) perform the actual tree surgery by removing from the old parent and adding to the new one. What exceptions do you raise for each invalid condition?

### My Answer

In my previous signature, the to-be-moved item name is part of the source and destination path. User can enter different names in the source path and destination path, and we have no way to tell if it is a mistake or user actually wants to rename the item while doing the move. 

To avoid this ambiguity, I change the method signature so that the name for the item-to-be-moved is moved to a separate parameter, while the source and destination paths are the container folders only.  

The move method will perform these checks:
- Move item is not the root folder
- Both the source and destination folder exist and are directories. If not, raise error.
- Source and destination folder are the same. If yes, return without taking any action.
- Destination folder is not a descendant of the move item (only applies if item is a folder). If yes, raise error.
- No name collision in the destination folder. If there is, raise error.

The following sequence should be performed as one atomic action (using a lock). This prevents any changes to the items involved while move is happening:
- get references to the source, destination folders and item-to-be-moved
- check name collision in the destination folder
- add the item to the destination's children list
- remove the item from the source's children list

### Feedback

What went well

Your check order is solid — you validate root, resolve both paths, confirm the destination is a directory, detect the descendant cycle, and check for name collisions before touching the tree. 
You correctly do the tree surgery by calling addChild() on the destination and removeChild() on the source, and you reused _getParentAndChild() to resolve paths cleanly. 
Nice reasoning in moving the item name into its own parameter to remove the rename-vs-move ambiguity. 
What needs improvement

Your cycle check destinationDirectory.startswith(sourceDirectory + itemName) can give false positives — a sibling like /x/ab starts with /x/a, so you should compare against the full source path with a trailing / to only match true descendants. 

Minor suggestions

Creating a threading.Lock() fresh inside move() gives no real protection since each call gets its own lock — a shared lock on the file system instance would be needed if you want atomicity. 




In [ ]:
    def move(self, itemName: str, sourceDirectory: str, destinationDirectory: str):
        if itemName == self.ROOT_PATH:
            raise RootDirectoryError

        if sourceDirectory == destinationDirectory:
            return  
        
        if sourceDirectory[-1] != "/":
            sourceDirectory += "/"
        
        lock = threading.Lock()
        with lock:
            source, move_item = self._getParentAndChild(sourceDirectory + itemName)
            _, destination = self._getParentAndChild(destinationDirectory)
            
            if not destination.isDirectory():
                raise ExpectedDirectoryError(destination.getName())
            
            # make sure not moving a folder to its own child
            if move_item.isDirectory() and destinationDirectory.startswith(sourceDirectory + itemName):
                raise InvalidMoveError
                    
            if destination.hasChild(itemName):
                raise NameConflictError(itemName)
                
            destination.addChild(move_item)
            source.removeChild(itemName)

Implement rename(newName, path). Beyond updating the entry's _name, what else needs to change in the parent Folder's _children map, and why? Walk through the steps in order.

### My Answer

In the parenet Folder's children map, I need to update the key for the entry, since I am using the name as key. 
Here are the steps for the method, steps 2 to 6 should be performed within lock:
1. Check item to rename is not the root, and new name is valid
2. Get reference to the entry and its parent folder
3. Check no collision for the new name 
4. Remove the entry from the parent's children map
5. Change the name of the entry
6. Add the entry to the parent's children map with the new name

### Feedback

What went well

Your rename() correctly does remove-then-re-add on the parent's _children map, which is exactly right since the map is keyed by name and just mutating _name would leave a stale key. 
You handle the important edge cases up front: rejecting root rename, validating the new name, and checking for collisions before mutating anything. 

Wrapping the mutation steps in a lock and ordering them cleanly (remove, setName, add) keeps the tree consistent. 

Minor suggestions

Consider a no-op short-circuit when newName equals the current name, so a self-rename doesn't falsely trip the collision check. 

In [ ]:
    def rename(self, newName: str, path: str):
        # Check item is not the root, and new name is valid
        if path == self.ROOT_PATH:
            raise RootDirectoryError
        
        if not self._isValidName(newName):
            raise InvalidNameError

        with self._lock:
            parent, child = self._getParentAndChild(path)

            # Check no collision for the new name
            if parent.hasChild(newName):
                raise NameConflictError
            
            parent.removeChild(child.getName())
            child.setName(newName)
            parent.addChild(child)

# 5. Extensibility

### Question

Your move() method acquires a single self._lock for the entire operation, but rename() and other methods do the same. What happens if two threads simultaneously call move() — one moving folder A into folder B, and another moving folder B into folder A? Walk me through how your current locking strategy handles this, and how you'd redesign it if you needed finer-grained locking per directory node.

### My Answer
In the current locking strategy, when two threads acquire the lock at the same time, only one of them can get it and the other will have to wait until the first thread releases the lock. Here is what's going to happen in sequence:
1. The first thread gets the lock and is moving folder A into folder B
2. When the first thread finishes the operation and releases the lock, the second thread will get the lock.
3. When the second thread attempts to move folder B into folder A, it will get ItemNotExist error for folder A because folder A no longer exists in its original path. The operation will fail. 

If a finer-grained locking is needed, the FileSystem class can keep track of a mapping of locks for files and folders that are currently being worked on and using the path as key.
When an operation is performed on a directory, the thread will acquire the lock for that directory. 
Unlike a single global lock, this allows multiple threads to work on separate directories without blocking one another, while still preventing contention issues on the same directory.
The trade-off is the complexity introduced in acquiring and maintaining the locks, as well as preventing deadlocks. Also when reading and writing to the lock mapping, the mapping itself should be protected by a global lock.

In the above scenario where 2 threads moving folder A and folder B, deadlock can happen if not handled properly. The first thread can hold the lock for folder A while waiting for folder B's lock and the second thread has folder B's lock while waiting for folder A's lock. This kind of deadlock can be prevented by using consistent lock ordering (for example, always acqure the locks by alphabetic order of the paths).

In addition, when deleting a folder, make sure to acquire locks for the folder and all of its descendants before taking any action. If any one of the descendants lock is not available, the delete operation should either wait with a timeout or fail. 

When a folder or file is being deleted, moved or renamed, its corresponding lock should be deleted from the mapping. Or alternatively, locks that are no longer in use should be removed from mapping. 


### Feedback
What went well

You correctly identified that the single global self._lock prevents the two-thread scenario from causing data corruption — you traced through the execution clearly: Thread 1 acquires the lock, moves A into B, releases it; Thread 2 then fails with ItemNotExistError because A is no longer at its original path. That's exactly right, and it shows you understand why the coarse-grained lock is actually safe, even if not maximally concurrent. 

Your fine-grained locking design was impressively thorough for a junior engineer — you identified the need for a lock map keyed by path, noted that the map itself needs a global lock, recognized the classic deadlock scenario with two threads holding one lock and waiting for the other, and proposed the correct solution (consistent lock ordering by path). You also handled the cleanup concern (removing locks after delete/rename), which many candidates miss entirely. 

Minor suggestions

One small addition: with per-directory locks, you'd also want to think about how to handle the 'descendant check' in move() — since that check reads the path string rather than traversing live nodes, it's less affected by fine-grained locking, but it's worth noting that any lock-ordering scheme needs to account for acquiring both the source parent and destination parent locks together. 

### Question

Your _getParentAndChild() method traverses the path by splitting on / and walking the tree node by node each time it's called. If you needed to add a find(name) method that returns all paths matching a given filename across the entire file system, how would your current design handle that, and what changes to your class structure would you make to support efficient name-based lookup?

### My Answer

To implement find(name), the curent implementation will have to go through all files and folders in the entire system, like a depth-first search graph traversal. The time complexity is O(n) and that would be inefficient. To improve that, I can use hash table or multi-map where the key will be the filename and value will be a set with all the paths that contain the filename (since the same filename can exist in multiple places). This allows for O(1) lookup time. However, the tradeoff is adding complexity and extra maintanence. When creating, deleting, renaming and moving FileSystemItems, the map needs to be updated accordingly. 
Here is a breakdown of the tasks needed:
- When creating a new FileSystemItem, if the filename already exist in the map, add the path to its value. If not, create a new entry in the map.
- When renaming a FileSystemItem, look up the current path in the map by the item's original name and remove it. Add the path to the map under the item's new name. If the item has descendants, all the paths for the descendants will need to be updated also.
- When moving an item, need to update its path in the map. And same apply to all its descendants.
- When deleting an item, need to delete its path from the map, as well as the paths for all its descendants.

As the system grows with more files and folders, maintaining all these mappings are not efficient too.
A more scalable approach will be to have an unique ID for each FileSystemItem, and use IDs instead of path in the map. There will be another mapping that maps the IDs to the actual FileSystemItems. Changes to the FileSystemItem class will include an ID property, a reference to its own parent folder and a method to generate its path by backtracking to its ancestors.    

### Feedback

What went well

You gave a thorough and accurate answer — you correctly identified that the current tree traversal approach is O(n) and explained exactly why: it requires a full DFS through all nodes. More importantly, you proposed a concrete solution (a name-to-paths index map in FileSystem) and clearly reasoned through the maintenance burden on every operation that modifies the file system (create, rename, move, delete). That's exactly the right way to think about the tradeoff. 

Your insight about using unique IDs instead of paths as the map's values was genuinely impressive for this level. You recognized that path strings are fragile — they break when items are renamed or moved — and proposed a more stable indirection layer (ID → item, plus a parent reference for path reconstruction). That's a scalable, real-world approach that shows strong systems thinking. 

Minor suggestions

One small thing to consider: you mentioned that renaming requires updating all descendant paths in the map — this is the key pain point with path-based indexing, and it's why the ID-based approach you described at the end is so much cleaner. If you ever get pushed on this in a real interview, you can use that argument to justify skipping the path-based approach entirely and going straight to IDs. 

### Question

Your move() method detects ancestor cycles by checking destinationDirectory.startswith(sourceDirectory + itemName + '/') using raw string comparison. Can you think of edge cases where this string-based check might give a wrong answer? How would you make this check more robust, and would you need to change anything in how Folder or FileSystemItem are structured to support a cleaner solution?

### My Answer
String comparison could go wrong when the two strings are different but they actually resolve to the same path. 
For example, having two consecutive '/' in one, but not the other. Or one ends in '/' and the other one does not.
Another edge case could be the string comparison not case-sensitive, while the file system is case sensitive. Thus, giving wrong results in the comparison. 

A better approach would be resolving the two paths to actual FileSystemItem objects and compare. The sequence will be like this:
- Resolve source path to actual folder.
- Split the destination path into folder names by '/'. Starting from the root, resolve the folder name to the actual folder by looking up from the _children map. 
- Check to see if the folder is same as the source folder. Repeat for all folders on the destination path. If any of the folders (except the last one) in the destination path is same as the source folder, that means we are copying a folder to its own descendant, which should be rejected. If we reach the last folder in the path and it is same as the source folder, no moving is needed (source and destination are the same). Otherwise, we can proceed with the move.    

No changes are needed on the current implementation.

### Feedback

What went well

You correctly identified real edge cases with the string-based check — consecutive slashes, trailing slash mismatches, and case sensitivity issues are genuine gotchas that many candidates miss. Good instinct to think about path normalization. 

Your proposed solution of resolving paths to actual FileSystemItem objects and walking the destination path step-by-step is solid and would actually work. Walking through _children lookups instead of string comparison is exactly the right direction, and you correctly noted that no changes to Folder or FileSystemItem are needed since the traversal logic can live in FileSystem. 

Minor suggestions

One small thing to consider: your traversal approach naturally gives you the resolved Folder objects to compare, but you could make it even cleaner by adding a private helper like _isAncestor(ancestor: Folder, descendant: Folder) -> bool in FileSystem that walks up from the destination using parent references — though that would require adding a _parent pointer to FileSystemItem. It's optional, but worth knowing that pattern exists.

# Key Takeaways

### Requirements
You demonstrated strong API design and error handling instincts from the start, and quickly filled in the two specific gaps around root directory protection and missing parent path validation when given feedback.

Always protect the root directory as a special case in file system designs. Operations like delete, move, and rename should throw a specific exception when the target is root, because root has no parent and removing it would destroy the entire tree structure.

When creating a file or directory at a given path, validate that the parent directory exists first. If a user tries to create '/a/b/c' but '/a/b' does not exist, throw a specific exception like ParentNotFoundException. This is a distinct error from a name collision and should be handled separately.

In a file system design, files should explicitly store string content as part of their data model. This small detail clarifies the difference between a File class and a Directory class during class design, and interviewers will often ask what a file actually holds when you move from requirements to class definitions.

### Entities
You demonstrated solid understanding of the core entities and ownership chain from the start, and after a small nudge you correctly identified how File and Directory can share a base type to support a unified children list in Directory.

When two classes share common properties like 'name', consider giving them a shared base class or interface. This lets you store both types in a single list. For example, a Directory's children list can be typed as List<FileSystemItem> where both File and Directory extend or implement FileSystemItem.

The Composite pattern is the right tool when you need a tree structure where individual items and groups of items are treated the same way. A File is a leaf node and a Directory is a composite node, but both implement the same interface so callers do not need to care which one they are working with.

A facade class like FileManager acts as the single entry point for all operations. It owns the root Directory, enforces validation, and hides the internal tree structure from callers. This keeps your API clean and puts constraint logic in one place instead of scattered across File and Directory.

### Class Design
You demonstrated a solid grasp of the Composite pattern and facade design, but missed a key structural detail where tree nodes need a parent pointer to reconstruct their own path and simplify operations like move and rename.

In tree structures like a file system, each node should hold a parent pointer back to its containing parent node. This lets any node reconstruct its own full path by walking up the tree without needing an external lookup table. Without it, the system managing the tree has to track parent relationships separately, which adds complexity and makes operations like move and rename harder to implement cleanly.

When you maintain a bidirectional link between a parent and child node, both sides of the link must be updated together. When adding a child, set the child's parent pointer to the new parent. When removing a child, clear the child's parent pointer back to null. Forgetting the cleanup step in remove is a common bug that leaves stale references and can cause incorrect path calculations.

### Implementation
You demonstrated strong structural thinking across path resolution and tree mutation, but had a few precise technical slip-ups around Python error raising, string-based path comparison, and shared locking that are worth locking in before your real interview.

In Python you must use 'raise' to signal an exception, not 'return'. Writing 'return InvalidPathError' hands back the class object as a value and silently continues execution. Always write 'raise InvalidPathError(...)' so the error actually interrupts the flow.

When checking if one path is a descendant of another, always append a trailing slash before comparing. Checking 'destinationPath.startswith(sourcePath)' will falsely match siblings like '/x/ab' when the source is '/x/a'. The correct check is 'destinationPath.startswith(sourcePath + "/")' so you only match true children.

A threading.Lock() created inside a method gives zero thread safety because every call gets its own fresh lock object. To protect shared state, create one lock as an instance variable on the class (e.g. self._lock = threading.Lock() in __init__) and acquire that same shared lock inside every method that mutates state.

When a data structure is keyed by name (like a dict mapping name to child node), mutating the name field directly leaves a stale key in the map. The correct pattern is remove the old key first, update the name, then re-insert under the new key. This keeps the map consistent and is the right approach for any rename operation on a keyed collection.

### Extensibility
You demonstrated strong systems thinking across all three attempts, consistently arriving at correct and sophisticated solutions for concurrency, indexing, and path validation in a file system design.

When using fine-grained locking with a lock map, always acquire multiple locks in a consistent order (e.g., sorted by path string) to prevent deadlocks. Two threads each holding one lock and waiting for the other is the classic deadlock scenario, and sorted acquisition breaks the cycle.

Path strings are fragile keys in any index or map because they break when items are renamed or moved. A more robust pattern is to assign each item a stable unique ID, store a map of ID to item, and add a parent reference to each item so you can reconstruct the current path on demand.

Checking if one folder is an ancestor of another by comparing path strings is error-prone due to edge cases like trailing slashes, consecutive slashes, and case sensitivity. The safer approach is to walk the actual tree nodes using parent references with a helper like _isAncestor(ancestor, descendant) that traverses up the tree structurally instead of comparing strings.

A name-to-paths index speeds up search from O(n) to O(1) but creates a maintenance burden. Every operation that modifies the file system (create, rename, move, delete) must update the index. Renaming is especially painful with path-based keys because all descendant paths in the index must also be updated, which is why ID-based indexing is cleaner in practice.



# Final Implementation

In [ ]:
from enum import Enum
from abc import ABC, abstractmethod
from pathlib import Path
import threading

class ItemType(Enum):
    FILE = 0
    FOLDER = 1

class FileSystemMetadata(ABC):
    def __init__(self, name: str, type: ItemType):
        super().__init__()
        self._name = name
        self._type = type

    def getName(self) -> str:
        return self._name

    def getType(self) -> ItemType: 
        return self._type

    @abstractmethod
    def print(self):
        pass

class FileMetadata(FileSystemMetadata):
    def __init__(self, name: str, content: str):
        super().__init__(name, ItemType.FILE)
        self._content = content

    def getContent(self) -> str:
        return self._content

    def print(self):
        print(f"{ItemType.FILE.name} : {self._name} [Content: {self._content}]")
    
class FolderMetadata(FileSystemMetadata):
    def __init__(self, name: str, childCount: int):
        super().__init__(name, ItemType.FOLDER)
        self._childCount = childCount

    def getChildCount(self) -> int:
        return self._childCount

    def print(self):
        print(f"{ItemType.FOLDER.name} : {self._name} [Child Count: {self._childCount}]")

class FileSystemItem(ABC):
    def __init__(self, name: str):
        super().__init__()
        self._name = name
        
    def getName(self) -> str:
        return self._name

    def setName(self, name: str):
        self._name = name

    @abstractmethod
    def getMetadata(self) -> FileSystemMetadata:
        pass

    @abstractmethod
    def isDirectory(self) -> bool:
        pass
    
class File(FileSystemItem):
    def __init__(self, name: str, content: str):
        super().__init__(name)
        self._content = content

    def getContent(self) -> str:
        return self._content
    
    def setContent(self, content:str):
        self._content = content

    def getMetadata(self):
        return FileMetadata(self._name, self._content)

    def isDirectory(self):
        return False
    
class Folder(FileSystemItem):
    def __init__(self, name):
        super().__init__(name)
        self._children = {}

    def addChild(self, item: FileSystemItem):
        self._children[item.getName()] = item

    def getChild(self, name: str) -> FileSystemItem:
        return self._children[name]
    
    def getChildren(self) -> list[FileSystemItem]:
        return list(self._children.values())
    
    def hasChild(self, name: str) -> bool:
        return name in self._children

    def removeChild(self, name: str):
        del self._children[name]

    def getMetadata(self):
        return FolderMetadata(self._name, len(self._children))
    
    def isDirectory(self):
        return True

class ItemNotExistError(Exception):
    def __init__(self, name: str):
        super().__init__(f"Item '{name}' does not exist. Make sure path is correct.")
    
class NameConflictError(Exception):
    def __init__(self, name: str):
        super().__init__(f"Item with '{name}' already exist.")

class InvalidPathError(Exception):
    def __init__(self):
        super().__init__("Path is not valid. It must start from the root directory.")
    
class InvalidMoveError(Exception):
    def __init__(self):
        super().__init__("Cannot move folder to its own subfolders.")

class InvalidNameError(Exception):
    def __init__(self):
        super().__init__("Name is empty or contains invalid characters.")

class ExpectedDirectoryError(Exception):
    def __init__(self, name: str):
        super().__init__(f"Item '{name}' is not a directory.")

class RootDirectoryError(Exception):
    def __init__(self):
        super().__init__("Cannot rename, move or delete the root directory.")

class FileSystem:
    ROOT_PATH = "/"

    def __init__(self):
        self._root = Folder(self.ROOT_PATH)
        self._lock = threading.Lock()

    def creatFile(self, filename: str, directory: str, content: str = None) -> FileMetadata:
        _, parent_folder = self._getParentAndChild(directory)
        if not parent_folder.isDirectory():
            raise InvalidPathError
        
        if parent_folder.hasChild(filename):
            raise NameConflictError

        newFile = File(filename, content)
        parent_folder.addChild(newFile)
        return newFile.getMetadata()

    def createFolder(self, foldername:str , directory:str) -> FolderMetadata:
        _, parent_folder = self._getParentAndChild(directory)
        if not parent_folder.isDirectory():
            raise InvalidPathError
                
        if parent_folder.hasChild(foldername):
            raise NameConflictError
        
        newFolder = Folder(foldername)
        parent_folder.addChild(newFolder)
        return newFolder.getMetadata()

    def delete(self, path: str):
        pass

    def _getParentAndChild(self, path: str) -> tuple[Folder, FileSystemItem]:
        if path == self.ROOT_PATH: # root has no parent
            return None, self._root
                
        items = path.strip("/").split("/")
        if not items:
            raise InvalidPathError
        
        parent = self._root
        for i, name in enumerate(items):
            name = name.strip()
            if not name:
                raise InvalidPathError
            if not parent.isDirectory():
                raise InvalidPathError
            if not parent.hasChild(name):
                raise ItemNotExistError(name)
            
            if i == len(items)-1:
                return parent, parent.getChild(name)
            parent = parent.getChild(name)

        raise ItemNotExistError(path)


    def get(self, path: str) -> FileSystemMetadata:
        _, child = self._getParentAndChild(path)
        return child.getMetadata()

    def _isValidName(self, name: str) -> bool:
        # not yet implement: alphanumeric characters, hyphens, underscores, and dots — no spaces or special characters like slashes
        return name != self.ROOT_PATH 


    def list(self, path: str) -> list[FileSystemMetadata]:
        _, folder = self._getParentAndChild(path)
        if not folder.isDirectory():
            raise ExpectedDirectoryError(folder.getName())
        return [item.getMetadata() for item in folder.getChildren()]

    def move(self, itemName: str, sourceDirectory: str, destinationDirectory: str):
        if itemName == self.ROOT_PATH:
            raise RootDirectoryError

        if sourceDirectory == destinationDirectory:
            return  
        
        if sourceDirectory[-1] != "/":
            sourceDirectory += "/"
        
        with self._lock:
            source, move_item = self._getParentAndChild(sourceDirectory + itemName)
            _, destination = self._getParentAndChild(destinationDirectory)
            
            if not destination.isDirectory():
                raise ExpectedDirectoryError(destination.getName())
            
            # make sure not moving a folder to its own child
            if move_item.isDirectory() and destinationDirectory.startswith(sourceDirectory + itemName + "/"):
                raise InvalidMoveError
                    
            if destination.hasChild(itemName):
                raise NameConflictError(itemName)
                
            destination.addChild(move_item)
            source.removeChild(itemName)

    def rename(self, newName: str, path: str):
        # Check item is not the root, and new name is valid
        if path == self.ROOT_PATH:
            raise RootDirectoryError
        
        if not self._isValidName(newName):
            raise InvalidNameError

        with self._lock:
            parent, child = self._getParentAndChild(path)

            # new name is not same as old name
            if child.getName() == newName:
                return
            
            # Check no collision for the new name
            if parent.hasChild(newName):
                raise NameConflictError
            
            parent.removeChild(child.getName())
            child.setName(newName)
            parent.addChild(child)




In [100]:
## TEST: /home/user/docs/file.txt
fs = FileSystem()
newfolder = fs.createFolder("home", "/")
newfolder = fs.createFolder("user", "/home")
newfolder = fs.createFolder("docs", "/home/user")
newfile = fs.creatFile("file.txt", "/home")
fs.rename("newHome", "/" )

path = "/"
f_list = fs.list(path)
print(f"items in {path}")
print("==========================")
for item in f_list:
    item.print()
print("")

path = "/newHome"
f_list = fs.list(path)
print(f"items in {path}")
print("==========================")
for item in f_list:
    item.print()
print("")



RootDirectoryError: Cannot rename, move or delete the root directory.